# 1. Skeleton e coeficientes com fatores RAW e TANH

A AST é sorteada com somas, produtos, constantes e lags. Para cada amostra, uma escolha reproduzível determina se `product_lags` envolve em `tanh` os lags sob produtos de fatores dependentes do passado. O encoder recebe a AST escolhida e o decoder aprende a forma ordenada **dessa mesma AST**. Essa escolha é nossa implementação para satisfazer condições estudadas por Shao e Wu (2007, Theorem 5.1). O gerador acrescenta depois um fator de escala ao mecanismo determinístico e uma inovação separada à série.

Cada termo é uma sequência ordenada de fatores RAW `(série,lag)` ou TANH `(série,lag,"tanh")`. A ordem é quantidade de fatores, depois série, lag e RAW antes de TANH. Repetições são preservadas. O grau máximo é quatro.


In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').exists())
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from tsfmxai.representations import Add,Multiply,Parameter,Lag,Tanh,MechanismBinding,canonicalize_polynomial,format_canonical_polynomial,format_monomial,walk
from tsfmxai.generators import saturate_product_lags,RandomTreeConfig,sample_random_mechanism,derive_sample_seed,CombinatorialTreeConfig,CombinatorialTreeSampler
from tsfmxai.generators.random_expression import SampledMechanism
from tsfmxai.symbolic_regression.canonical_data import CanonicalCodec,select_canonical_mechanism
pd.set_option('display.max_colwidth',120)

def mul(*nodes):
    out=nodes[0]
    for node in nodes[1:]:out=Multiply(out,node)
    return out
def add(*nodes):
    out=nodes[0]
    for node in nodes[1:]:out=Add(out,node)
    return out
def case(label,why,expr,parameters={},lags={}):
    binding=MechanismBinding(parameters,lags)
    raw=canonicalize_polynomial(expr,binding)
    actual=saturate_product_lags(expr)
    poly=canonicalize_polynomial(actual,binding)
    return dict(label=label,why=why,input=expr,actual=actual,binding=binding,raw=raw,poly=poly)
x=Lag(0,'x');y=Lag(1,'y');z=Lag(2,'z');x2=Lag(0,'x2')
cases=[
case('Constante','termo independente',Parameter('c'),{'c':2}),
case('RAW','lag isolado',x,lags={'x':1}),
case('Coeficiente','constante vezes lag',mul(Parameter('c'),x),{'c':-.7},{'x':1}),
case('Produto','dois lags protegidos',mul(x,y),lags={'x':1,'y':2}),
case('A=(x+2)y','proteção antes da distributiva',mul(add(x,Parameter('c')),y),{'c':2},{'x':1,'y':2}),
case('B=xy+2y','termo externo permanece RAW',add(mul(x,y),mul(Parameter('c'),y)),{'c':2},{'x':1,'y':2}),
case('Quadrado','lag repetido',mul(x,x),lags={'x':1}),
case('Três fatores','ordem dos fatores',mul(z,y,x),lags={'x':1,'y':2,'z':3}),
case('Quatro fatores','grau máximo',mul(x,y,z,x2),lags={'x':1,'x2':2,'y':2,'z':3}),
case('RAW e TANH','mesmo lag em dois papéis',add(x,mul(x,y)),lags={'x':1,'y':2}),
case('Cancelamento','suporte realizado vazio',add(mul(x,y),mul(Parameter('n'),mul(y,x))),{'n':-1},{'x':1,'y':2}),
case('Duas somas','constante, simples e interação',mul(add(x,Parameter('c')),add(y,Parameter('d'))),{'c':2,'d':3},{'x':1,'y':2})
]
display(pd.DataFrame([{'Caso':q['label'],'Motivação':q['why'],'Entrada':str(q['input']),
 'Sem proteção':format_canonical_polynomial(q['raw']),
 'Após proteção':format_canonical_polynomial(q['poly']),
 'Skeleton RAW':q['raw'].skeleton,'Skeleton protegido':q['poly'].skeleton,
 'Coeficientes RAW':q['raw'].coefficients,'Coeficientes protegidos':q['poly'].coefficients} for q in cases]))
assert cases[4]['poly'].canonical_key!=cases[5]['poly'].canonical_key


,Caso,Motivação,Entrada,Sem proteção,Após proteção,Skeleton RAW,Skeleton protegido,Coeficientes RAW,Coeficientes protegidos
0,Constante,termo independente,Parameter(slot='c'),2,2,"((),)","((),)","(2.0,)","(2.0,)"
1,RAW,lag isolado,"Lag(variable=0, slot='x')",(1)*s_0[t-1],(1)*s_0[t-1],"(((0, 1),),)","(((0, 1),),)","(1.0,)","(1.0,)"
2,Coeficiente,constante vezes lag,"Multiply(left=Parameter(slot='c'), right=Lag(variable=0, slot='x'))",(-0.7)*s_0[t-1],(-0.7)*s_0[t-1],"(((0, 1),),)","(((0, 1),),)","(-0.7,)","(-0.7,)"
3,Produto,dois lags protegidos,"Multiply(left=Lag(variable=0, slot='x'), right=Lag(variable=1, slot='y'))",(1)*s_0[t-1] * s_1[t-2],(1)*tanh(s_0[t-1]) * tanh(s_1[t-2]),"(((0, 1), (1, 2)),)","(((0, 1, tanh), (1, 2, tanh)),)","(1.0,)","(1.0,)"
4,A=(x+2)y,proteção antes da distributiva,"Multiply(left=Add(left=Lag(variable=0, slot='x'), right=Parameter(slot='c')), right=Lag(variable=1, slot='y'))",(2)*s_1[t-2] + (1)*s_0[t-1] * s_1[t-2],(2)*tanh(s_1[t-2]) + (1)*tanh(s_0[t-1]) * tanh(s_1[t-2]),"(((1, 2),), ((0, 1), (1, 2)))","(((1, 2, tanh),), ((0, 1, tanh), (1, 2, tanh)))","(2.0, 1.0)","(2.0, 1.0)"
5,B=xy+2y,termo externo permanece RAW,"Add(left=Multiply(left=Lag(variable=0, slot='x'), right=Lag(variable=1, slot='y')), right=Multiply(left=Parameter(sl...",(2)*s_1[t-2] + (1)*s_0[t-1] * s_1[t-2],(2)*s_1[t-2] + (1)*tanh(s_0[t-1]) * tanh(s_1[t-2]),"(((1, 2),), ((0, 1), (1, 2)))","(((1, 2),), ((0, 1, tanh), (1, 2, tanh)))","(2.0, 1.0)","(2.0, 1.0)"
6,Quadrado,lag repetido,"Multiply(left=Lag(variable=0, slot='x'), right=Lag(variable=0, slot='x'))",(1)*s_0[t-1] * s_0[t-1],(1)*tanh(s_0[t-1]) * tanh(s_0[t-1]),"(((0, 1), (0, 1)),)","(((0, 1, tanh), (0, 1, tanh)),)","(1.0,)","(1.0,)"
7,Três fatores,ordem dos fatores,"Multiply(left=Multiply(left=Lag(variable=2, slot='z'), right=Lag(variable=1, slot='y')), right=Lag(variable=0, slot=...",(1)*s_0[t-1] * s_1[t-2] * s_2[t-3],(1)*tanh(s_0[t-1]) * tanh(s_1[t-2]) * tanh(s_2[t-3]),"(((0, 1), (1, 2), (2, 3)),)","(((0, 1, tanh), (1, 2, tanh), (2, 3, tanh)),)","(1.0,)","(1.0,)"
8,Quatro fatores,grau máximo,"Multiply(left=Multiply(left=Multiply(left=Lag(variable=0, slot='x'), right=Lag(variable=1, slot='y')), right=Lag(var...",(1)*s_0[t-1] * s_0[t-2] * s_1[t-2] * s_2[t-3],(1)*tanh(s_0[t-1]) * tanh(s_0[t-2]) * tanh(s_1[t-2]) * tanh(s_2[t-3]),"(((0, 1), (0, 2), (1, 2), (2, 3)),)","(((0, 1, tanh), (0, 2, tanh), (1, 2, tanh), (2, 3, tanh)),)","(1.0,)","(1.0,)"
9,RAW e TANH,mesmo lag em dois papéis,"Add(left=Lag(variable=0, slot='x'), right=Multiply(left=Lag(variable=0, slot='x'), right=Lag(variable=1, slot='y')))",(1)*s_0[t-1] + (1)*s_0[t-1] * s_1[t-2],(1)*s_0[t-1] + (1)*tanh(s_0[t-1]) * tanh(s_1[t-2]),"(((0, 1),), ((0, 1), (1, 2)))","(((0, 1),), ((0, 1, tanh), (1, 2, tanh)))","(1.0, 1.0)","(1.0, 1.0)"


A e B são iguais por distributividade antes da proteção. Suas árvores protegidas são mecanismos distintos: o termo `2y` de A contém TANH; o de B contém RAW. Esta diferença é visível no skeleton e nos coeficientes.

In [2]:
from tsfmxai.generators.trajectory import evaluate_bound_expression
codec=CanonicalCodec(num_series=3,minimum_lag=1,maximum_lag=3)
rng=np.random.default_rng(20261101)
rows=[]
for i,q in enumerate(cases):
    sample=SampledMechanism(q['input'],q['binding'],20261101,i)
    for probability in (0.,1.):
        chosen,selected=select_canonical_mechanism(sample,probability=probability,
            root_seed=20261101,sample_index=i)
        expected=q['poly'] if selected else q['raw']
        encoded=codec.encode(chosen)
        decoded=codec.decode(encoded['target'][1:],encoded['coefficient_targets'][1:])
        histories={j:rng.normal(size=12) for j in range(3)}
        direct=evaluate_bound_expression(chosen.expression,chosen.binding,time=12,histories=histories)
        expanded=expected.evaluate(histories,time=12)
        assert decoded['valid'] and decoded['skeleton']==expected.skeleton
        assert decoded['coefficients']==expected.coefficients
        rows.append({'Caso':q['label'],'Proteção selecionada':selected,'Nós TANH':encoded['tanh_count'],
                     'Tokens':codec.target_text(encoded['target']),'Roundtrip':decoded['valid'],
                     'Erro numérico':abs(direct-expanded)})
display(pd.DataFrame(rows))
assert max(r['Erro numérico'] for r in rows)<1e-12


,Caso,Proteção selecionada,Nós TANH,Tokens,Roundtrip,Erro numérico
0,Constante,False,0,BOS END_TERM EOS,True,0.000000e+00
1,Constante,True,0,BOS END_TERM EOS,True,0.000000e+00
2,RAW,False,0,"BOS LAG(0, 1) END_TERM EOS",True,0.000000e+00
3,RAW,True,0,"BOS LAG(0, 1) END_TERM EOS",True,0.000000e+00
4,Coeficiente,False,0,"BOS LAG(0, 1) END_TERM EOS",True,0.000000e+00
5,Coeficiente,True,0,"BOS LAG(0, 1) END_TERM EOS",True,0.000000e+00
6,Produto,False,0,"BOS LAG(0, 1) LAG(1, 2) END_TERM EOS",True,0.000000e+00
7,Produto,True,2,"BOS LAG(0, 1, 'tanh') LAG(1, 2, 'tanh') END_TERM EOS",True,0.000000e+00
8,A=(x+2)y,False,0,"BOS LAG(1, 2) END_TERM LAG(0, 1) LAG(1, 2) END_TERM EOS",True,0.000000e+00
9,A=(x+2)y,True,2,"BOS LAG(1, 2, 'tanh') END_TERM LAG(0, 1, 'tanh') LAG(1, 2, 'tanh') END_TERM EOS",True,3.469447e-18


O roundtrip verifica separadamente as duas escolhas para cada fórmula. A entrada RAW leva ao alvo RAW; a entrada protegida leva ao alvo protegido. A avaliação funcional adicional usa históricos sorteados e mede o arredondamento entre a AST escolhida e sua forma ordenada.

In [3]:
bad=[('Cinco fatores',mul(x,y,z,x2,x),MechanismBinding({}, {'x':1,'x2':2,'y':2,'z':3})),
     ('TANH de produto',Tanh(mul(x,y)),MechanismBinding({}, {'x':1,'y':2}))]
errors=[]
for label,expr,binding in bad:
    try: canonicalize_polynomial(expr,binding)
    except (TypeError,ValueError) as err:errors.append((label,str(err)))
assert len(errors)==2
display(pd.DataFrame(errors,columns=['Caso rejeitado','Motivo']))


,Caso rejeitado,Motivo
0,Cinco fatores,Expanded monomial has degree 5; max_degree=4.
1,TANH de produto,Only Tanh(Lag) is supported as a canonical factor.


Os limites da representação são explícitos: o canonizador aceita `Tanh(Lag)`, preserva até quatro fatores por termo e rejeita `Tanh` de uma expressão inteira.

In [4]:
from tsfmxai.generators.shao_wu import ShaoWuConfig,build_shao_wu_mechanism
single=Multiply(Lag(0,'a'),Lag(0,'b'))
single_binding=MechanismBinding({}, {'a':1,'b':2})
protected=saturate_product_lags(single)
certified=build_shao_wu_mechanism((protected,),(single_binding,),
    ShaoWuConfig(tree_config=RandomTreeConfig(num_series=1),lag_transform='product_lags'),seed=20261101)
final_poly=canonicalize_polynomial(certified.expressions[0],certified.bindings[0])
display(pd.DataFrame([{'Escala':certified.scale,'Ganho certificado':certified.certificate.total_gain,
    'AST final':str(certified.expressions[0]),'Forma ordenada':format_canonical_polynomial(final_poly),
    'Skeleton':final_poly.skeleton,'Coeficientes':final_poly.coefficients}]))
assert certified.certificate.holds


,Escala,Ganho certificado,AST final,Forma ordenada,Skeleton,Coeficientes
0,0.38,0.76,"Multiply(left=Parameter(slot='GMC_SCALE'), right=Multiply(left=Tanh(operand=Lag(variable=0, slot='a')), right=Tanh(o...",(0.38)*tanh(s_0[t-1]) * tanh(s_0[t-2]),"(((0, 1, tanh), (0, 2, tanh)),)","(0.38,)"


Este exemplo percorre também o escalonamento do gerador. Ele modifica os coeficientes da forma ordenada e conserva os fatores TANH do skeleton.

In [5]:
import time
from tsfmxai.representations import operator_count
timing=[]
for q in cases:
    started=time.perf_counter()
    for _ in range(5000):canonicalize_polynomial(q['actual'],q['binding'])
    timing.append({'Caso':q['label'],'Operações da AST':operator_count(q['actual']),
                   'Termos no skeleton':len(q['poly'].skeleton),
                   'Tempo médio (µs)':1e6*(time.perf_counter()-started)/5000})
display(pd.DataFrame(timing))


,Caso,Operações da AST,Termos no skeleton,Tempo médio (µs)
0,Constante,0,1,30.71712
1,RAW,0,1,21.04492
2,Coeficiente,1,1,39.34672
3,Produto,3,1,44.57086
4,A=(x+2)y,4,2,73.23552
5,B=xy+2y,5,2,84.39910
6,Quadrado,3,1,48.20820
7,Três fatores,5,1,58.81864
8,Quatro fatores,7,1,68.91890
9,RAW e TANH,4,2,67.86720


O cronômetro mede 5.000 canonizações completas por fórmula, incluindo expansão, agrupamento e avaliação dos coeficientes. O tempo depende da estrutura e da quantidade de termos produzidos.

In [6]:
from dataclasses import replace
config=RandomTreeConfig(minimum_operators=1,maximum_operators=6,num_series=3,
    minimum_lag=1,maximum_lag=3,parameter_probability=.4,lag_probability=.6)
rng=np.random.default_rng(20261102)
checked=0;rejected=0;max_error=0.
for i in range(100):
    sample=sample_random_mechanism(config,seed=derive_sample_seed(20261102,i))
    actual=saturate_product_lags(sample.expression)
    try:poly=canonicalize_polynomial(actual,sample.binding)
    except ValueError:rejected+=1;continue
    for _ in range(10):
        histories={j:rng.normal(size=12) for j in range(3)}
        direct=evaluate_bound_expression(actual,sample.binding,time=12,histories=histories)
        max_error=max(max_error,abs(direct-poly.evaluate(histories,time=12)))
    checked+=1
display(pd.DataFrame([{'ASTs sorteadas':100,'Canonizadas':checked,'Rejeitadas':rejected,
                       'Históricos por AST':10,'Erro numérico máximo':max_error}]))
assert checked+rejected==100 and max_error<1e-10


,ASTs sorteadas,Canonizadas,Rejeitadas,Históricos por AST,Erro numérico máximo
0,100,99,1,10,8.881784e-16


O teste adicional sorteia ASTs multivariadas de três séries e compara cada função protegida com sua forma ordenada em dez históricos independentes. Rejeições por grau são contabilizadas explicitamente.

## Registro do experimento

**Pergunta:** o canonizador e o codec preservam fatores RAW/TANH, ordem e coeficientes? **Configuração:** 12 fórmulas avaliadas nas duas escolhas de proteção, 3 séries, lags 1–3, grau máximo 4; 5.000 transformações por fórmula protegida; 100 ASTs sorteadas, dez históricos por AST aceita. **Seeds:** 20261101 e 20261102. **Proveniência:** `src/tsfmxai/generators/shao_wu.py`, `src/tsfmxai/representations/canonical_polynomial.py`, `src/tsfmxai/symbolic_regression/canonical_data.py`. **Métricas:** roundtrip, erro numérico, rejeições e tempo médio.  **Resultados:** 24/24 pares AST-alvo passaram no roundtrip (12 fórmulas nas escolhas RAW e protegida); 99/100 ASTs sorteadas foram canonizadas, com diferença numérica máxima de 8,88×10⁻¹⁶ em dez históricos por AST aceita. Duas entradas deliberadas foram rejeitadas pelas regras da linguagem. **Resumo científico:** a escolha de proteção altera a AST de entrada e o alvo correspondente; uma escolha sem produto relevante mantém a expressão RAW. **Reprodução:** `.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace experiments/canonicalization/1_skeleton_and_coefficients.ipynb`.
